# RAG-Optimised Pipeline: PDF → ChromaDB → Keyword Extraction → SQL Plan

**Goal:** Replace the brute-force `pdf_text[:4000]` truncation with Retrieval-Augmented Generation.

**Baseline:** 316.7s / 36k tokens (gpt-5-mini) on `01_pdf_to_db_query.ipynb`.

**Approach:**
1. Chunk the PDF text with `RecursiveCharacterTextSplitter`
2. Embed chunks into an ephemeral ChromaDB collection (in-memory)
3. Retrieve relevant chunks via schema-aware similarity queries
4. Pass only retrieved context to the LLM nodes — smaller context = fewer tokens = faster

**Graph:** `START → node_retrieve → node_extract → node_plan → END`

## Observability + AI Platform Setup

In [ ]:
import os
import grpc
from opentelemetry.sdk.trace import TracerProvider, Resource
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from openinference.instrumentation.langchain import LangChainInstrumentor

if "tracer_provider" not in globals():
    space_id = os.environ["ARIZE_SPACE_ID"]
    api_key  = os.environ["ARIZE_API_KEY"]

    resource = Resource.create({
        "openinference.project.name": "fair-play-initiative-rag",
    })

    exporter = OTLPSpanExporter(
        endpoint="otlp.arize.com",
        headers=[
            ("authorization", api_key),
            ("api_key",       api_key),
            ("arize-space-id", space_id),
            ("space_id",       space_id),
            ("arize-interface", "otel"),
        ],
        credentials=grpc.ssl_channel_credentials(),
    )

    tracer_provider = TracerProvider(resource=resource)
    tracer_provider.add_span_processor(SimpleSpanProcessor(exporter))

    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
    print("Arize AX tracing initialized — project: fair-play-initiative-rag")
else:
    print("Arize AX already initialized — skipping")

print("View traces → https://app.arize.com")

In [ ]:
# ── Observability: Local Metrics Tracker ─────────────────────────────────────
import time
from langchain_core.callbacks import BaseCallbackHandler

class MetricsCallback(BaseCallbackHandler):
    def __init__(self):
        self.records = []
        self._start = {}

    def on_llm_start(self, serialized, prompts, *, run_id, **kw):
        self._start[run_id] = time.time()

    def on_chat_model_start(self, serialized, messages, *, run_id, **kw):
        self._start[run_id] = time.time()

    def on_llm_end(self, response, *, run_id, **kw):
        elapsed = (time.time() - self._start.pop(run_id, time.time())) * 1000
        usage = (response.llm_output or {}).get("token_usage", {})
        self.records.append({
            "run_id": str(run_id),
            "latency_ms": round(elapsed),
            "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": usage.get("completion_tokens"),
            "total_tokens": usage.get("total_tokens"),
        })

metrics_cb = MetricsCallback()
print("MetricsCallback ready")

## 1. Imports + Model Setup

In [ ]:
import os
import json
import fitz  # PyMuPDF
import sqlite3
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# LLM: gpt-5-mini via OpenRouter
model = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

# Embeddings: text-embedding-3-small via OpenRouter
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

print(f"LLM: gpt-5-mini | Embeddings: text-embedding-3-small")

In [ ]:
# ── Load Schema + System Prompt from files ────────────────────────────────────
with open("Schema.json") as f:
    FPI_SCHEMA = json.load(f)

with open("SystemPrompt.md") as f:
    KEYWORD_SYSTEM_PROMPT = f.read()

_schema_str = json.dumps(FPI_SCHEMA, indent=2)
KEYWORD_SYSTEM_MSG = f"{KEYWORD_SYSTEM_PROMPT}\n\n# FPI Schema\n\n```json\n{_schema_str}\n```"

print(f"Schema loaded: {len(FPI_SCHEMA)} tables — {[t['table'] for t in FPI_SCHEMA]}")
print(f"System prompt: {len(KEYWORD_SYSTEM_PROMPT)} chars")

## 2. Pydantic Output Schemas

In [ ]:
class KeywordExtraction(BaseModel):
    policy: list[str] = Field(default_factory=list, description="Policy-level keywords: codes, names, framework attributes, accrual model type, and scope.")
    organization: list[str] = Field(default_factory=list, description="Organization names and identifiers.")
    region: list[str] = Field(default_factory=list, description="Region names, jurisdictions, and applicable labor laws.")
    rule: list[str] = Field(default_factory=list, description="Rule names, tier definitions, conditions, point thresholds, and escalation criteria.")
    attendance_log: list[str] = Field(default_factory=list, description="Attendance event types, violation definitions, scheduled vs actual time terms, and log statuses.")
    point_history: list[str] = Field(default_factory=list, description="Point accrual, expiration, active status, and reason classification terms.")
    alert: list[str] = Field(default_factory=list, description="Alert types, automatic trigger conditions, escalation actions, and documentation levels.")
    other_relevant_terms: list[str] = Field(default_factory=list, description="Terms relevant to the policy but not directly mappable to a specific schema element.")
    confidence_score: float = Field(description="Confidence score of the extraction between 0.0 and 1.0.")


class TableUsage(BaseModel):
    table: str = Field(description="Exact table name from the FPI schema.")
    alias: str = Field(description="Short alias used in the SQL query (e.g. 'e' for employee).")
    purpose: str = Field(description="One sentence: why this table is needed for this query.")

class JoinStep(BaseModel):
    left: str = Field(description="Left-side table alias or name.")
    right: str = Field(description="Right-side table alias or name.")
    condition: str = Field(description="The ON clause expression, e.g. 'e.organization_id = o.id'.")
    join_type: str = Field(description="JOIN type: INNER, LEFT, RIGHT, or FULL.")

class FilterCondition(BaseModel):
    description: str = Field(description="Plain-English description of what this filter does.")
    expression: str = Field(description="The SQL expression for this filter.")
    source_keyword: str = Field(description="The extracted keyword that motivated this filter.")

class AggregationStep(BaseModel):
    column_name: str = Field(description="Output column alias for this aggregation.")
    expression: str = Field(description="The aggregate SQL expression.")
    purpose: str = Field(description="What business question this aggregation answers.")

class SQLQueryPlan(BaseModel):
    objective: str = Field(
        description="One sentence: what does this SQL operation accomplish? "
                    "For ingestion plans, describe what data is being inserted and into which tables. "
                    "Do NOT describe downstream analytics or reporting use cases."
    )
    tables_required: list[TableUsage] = Field(default_factory=list, description="All tables needed, with aliases and purpose.")
    joins: list[JoinStep] = Field(default_factory=list, description="All JOIN relationships required, in the order they should be applied.")
    where_filters: list[FilterCondition] = Field(default_factory=list, description="All WHERE-clause filters, each tied to a source keyword.")
    grouping_columns: list[str] = Field(default_factory=list, description="Column references to include in GROUP BY. Leave empty for ingestion plans.")
    aggregations: list[AggregationStep] = Field(default_factory=list, description="All aggregate columns to compute. Leave empty for ingestion plans.")
    having_filters: list[FilterCondition] = Field(default_factory=list, description="Post-aggregation HAVING filters. Leave empty for ingestion plans.")
    output_columns: list[str] = Field(default_factory=list, description="Final SELECT column names or aliases. Leave empty for ingestion plans.")
    uses_cte: bool = Field(description="True if the query requires a Common Table Expression.")
    cte_description: str | None = Field(default=None, description="Complete procedural plan with column mappings and rule enumerations.")
    confidence_score: float = Field(description="Confidence score between 0.0 and 1.0.")

print("Output schemas defined: KeywordExtraction, SQLQueryPlan + sub-models")

## 3. PDF Extraction

In [ ]:
PDF_PATH = "/input_files/Acme_Manufacturing_Attendance_Policy.pdf"

doc = fitz.open(PDF_PATH)
num_pages = len(doc)
pdf_text = ""
for page in doc:
    pdf_text += page.get_text()
doc.close()

print(f"Extracted {num_pages} pages, {len(pdf_text)} characters")
print(f"\n--- First 600 characters ---")
print(pdf_text[:600])

## 4. RAG: Chunk + Embed + Retrieve

This is the key optimisation. Instead of passing `pdf_text[:4000]` to the LLM,
we chunk the full document, embed into ChromaDB, and retrieve only the most
relevant sections using schema-aware queries.

In [ ]:
# ── Chunk the PDF text ─────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_text(pdf_text)
print(f"Split into {len(chunks)} chunks (800 chars, 200 overlap)")
print(f"Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")

In [ ]:
# ── Embed into ephemeral ChromaDB ──────────────────────────────────────────────
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="policy_chunks",
)
print(f"ChromaDB collection created: {vectorstore._collection.count()} vectors")

In [ ]:
# ── Retrieve relevant chunks using schema-aware queries ───────────────────────
# Multiple targeted queries ensure coverage across all schema entity types.
# Using generous k values — optimise later once baseline is established.

RAG_QUERIES = [
    "attendance policy rules violations points thresholds escalation corrective action",
    "organization company employer region location jurisdiction labor laws",
    "policy name effective date scope accrual model rolling window",
    "excused unexcused absence tardy no-call no-show early departure",
    "perfect attendance reduction milestone FMLA ADA PTO leave approved",
]

K_PER_QUERY = 8  # generous — retrieve top 8 per query

seen_content = set()
retrieved_chunks = []

for q in RAG_QUERIES:
    docs = vectorstore.similarity_search(q, k=K_PER_QUERY)
    for doc in docs:
        if doc.page_content not in seen_content:
            seen_content.add(doc.page_content)
            retrieved_chunks.append(doc.page_content)

rag_context = "\n\n---\n\n".join(retrieved_chunks)

print(f"Retrieved {len(retrieved_chunks)} unique chunks from {len(RAG_QUERIES)} queries (k={K_PER_QUERY})")
print(f"RAG context size: {len(rag_context)} chars (vs original {len(pdf_text)} chars)")
print(f"Compression ratio: {len(rag_context)/len(pdf_text):.1%}")
print(f"\n--- First 500 chars of RAG context ---")
print(rag_context[:500])

## 5. Keyword Extraction (Step 3a) — RAG-powered

Uses `rag_context` instead of `pdf_text[:4000]`.

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────────
_KEYWORD_FIELDS = [
    "policy", "organization", "region", "rule",
    "attendance_log", "point_history", "alert", "other_relevant_terms",
]

def _format_keywords(kw: KeywordExtraction) -> str:
    lines = []
    for field in _KEYWORD_FIELDS:
        values = getattr(kw, field, [])
        if values:
            lines.append(f"[{field}]")
            lines.extend(f"  \u2022 {v}" for v in values)
    return "\n".join(lines)


# ── Prompt + Model ─────────────────────────────────────────────────────────────
kw_model = model.with_structured_output(KeywordExtraction)

KEYWORD_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content=KEYWORD_SYSTEM_MSG),
    HumanMessagePromptTemplate.from_template("{pdf_text}"),
])

# ── Run: Step 3a — Keyword Extraction (RAG context) ───────────────────────────
try:
    keyword_data = (KEYWORD_PROMPT | kw_model).invoke(
        {"pdf_text": rag_context},
        config={"callbacks": [metrics_cb]},
    )
    print("--- Keyword Extraction Results (RAG-powered) ---")
    for field in _KEYWORD_FIELDS:
        values = getattr(keyword_data, field, [])
        if values:
            print(f"\n[{field}] ({len(values)} terms):")
            for kw in values:
                print(f"  \u2022 {kw}")
    print(f"\nConfidence: {keyword_data.confidence_score}")
except Exception as e:
    print(f"Error: {e}")
    keyword_data = None

## 6. Plan Prompt + Query Plan Generation (Step 3b) — RAG-powered

In [ ]:
# ── Plan Prompt Setup ─────────────────────────────────────────────────────────
PLAN_SYSTEM_MSG = (
    "You are a SQL ingestion planning assistant for the Fair Play Initiative (FPI). "
    "Your task is to analyse extracted policy keywords and devise a precise, "
    "structured plan for INSERT/UPSERT statements that load this policy document "
    "into the FPI database.\n\n"
    "WHAT YOU ARE PLANNING: policy document ingestion \u2014 not analytics or reporting. "
    "The goal is to persist the organisation, region, policy metadata, and rules "
    "extracted from the PDF into the appropriate FPI configuration tables.\n\n"
    "TARGET TABLES (the only tables this plan may touch):\n"
    "  \u2022 organization  \u2014 one row for the employer entity\n"
    "  \u2022 region        \u2014 one row for the jurisdiction\n"
    "  \u2022 organization_region \u2014 junction row linking the two\n"
    "  \u2022 policy        \u2014 one row for the policy document itself\n"
    "  \u2022 rule          \u2014 one row per violation tier / escalation rule\n\n"
    "OFF-LIMITS TABLES (do NOT plan any writes to these):\n"
    "  \u2022 employee, attendance_log, point_history, alert\n"
    "  These are populated at runtime by the HR system, never by policy ingestion.\n\n"
    "CRITICAL CONSTRAINT: Do NOT write any SQL. Do NOT include SQL fragments in "
    "any field. Your entire output must be a structured plan in plain English "
    "and column-name notation only.\n\n"
    "PLANNING GUIDANCE\n\n"
    "INSERT order \u2014 always respect FK constraints:\n"
    "  organization \u2192 region \u2192 organization_region \u2192 policy \u2192 rule\n\n"
    "INSERT strategy per table:\n"
    "  \u2022 organization        : INSERT OR IGNORE \u2014 master record; re-ingestion must not overwrite\n"
    "  \u2022 region              : INSERT OR IGNORE \u2014 shared infrastructure; same reason\n"
    "  \u2022 organization_region : INSERT OR IGNORE \u2014 junction row; no column data to overwrite\n"
    "  \u2022 policy              : INSERT OR REPLACE \u2014 a revised policy document supersedes prior versions; "
    "effective_date and active flag must stay current\n"
    "  \u2022 rule                : INSERT OR REPLACE \u2014 policy revisions may update thresholds and point values\n\n"
    "ID chaining \u2014 always set uses_cte = True and describe the full chain in cte_description:\n"
    "  After each INSERT, the generated ID flows as FK into subsequent INSERTs:\n"
    "  organization.id \u2192 region.id \u2192 organization_region.(organization_id, region_id) "
    "\u2192 policy.(organization_id, region_id) \u2192 policy.id \u2192 rule.policy_id\n"
    "  Without this chain, FK constraints cannot be satisfied.\n\n"
    "cte_description \u2014 this field carries the COMPLETE procedural plan. "
    "It MUST contain all of the following (gpt-5-mini: do not omit any section):\n"
    "  1. For each table in INSERT order, a numbered step containing:\n"
    "     - Table name and INSERT strategy (OR IGNORE / OR REPLACE)\n"
    "     - Every column to be populated, mapped to the exact extracted keyword that provides its value\n"
    "     - The natural key used for the existence check (matches the where_filters entry)\n"
    "  2. A complete enumeration of ALL rule rows in four groups \u2014 include every row "
    "from the policy; do not summarise or omit:\n"
    "     a. Violation/occurrence rules \u2014 one row per violation type (points > 0)\n"
    "     b. Approved-leave rules \u2014 one row per leave type (points = 0.0)\n"
    "     c. Perfect-attendance reduction rules \u2014 one row per milestone "
    "(threshold = consecutive clean days: 30, 60, 90, 365; points = negative value)\n"
    "     d. Corrective-action trigger rules \u2014 one row per level "
    "(threshold = lower bound of point range; points = 0.0)\n"
    "     Also include operational rule rows for: rolling 12-month window, negative balance floor, "
    "occurrence definition, excused/unexcused determination deadlines.\n"
    "     For each rule row provide: id (kebab slug), name, condition (plain-English HRIS trigger), "
    "threshold, points, description, active.\n"
    "  3. ID chaining summary (one line per FK relationship).\n"
    "  4. Operational reminders \u2014 placeholders that need resolution from HRIS/facility master data "
    "(region code, timezone, mandatory OT points confirmation, etc.).\n"
    "  Completeness is critical \u2014 a missing rule row means the policy is partially loaded.\n\n"
    "joins \u2014 for ingestion plans, use this field to describe FK dependency chains, NOT SQL joins:\n"
    "  \u2022 left: the child table (the one holding the FK column)\n"
    "  \u2022 right: the parent table (the one holding the PK)\n"
    "  \u2022 condition: which FK in the child references which PK in the parent "
    "(e.g. 'organization_region.organization_id = organization.id')\n"
    "  \u2022 join_type: always 'INNER' for required FK dependencies\n"
    "  List one entry per FK relationship, in dependency order.\n\n"
    "Rule rows \u2014 plan one row per:\n"
    "  \u2022 Each distinct violation type (NCNS 1st day, NCNS consecutive, unexcused full-day, "
    "unexcused half-day, tardy major, tardy minor, third minor tardy in month, "
    "early departure, holiday/critical day absence, plant shutdown no-return, "
    "mandatory OT no-show, voluntary OT no-show, and any other named in the policy)\n"
    "  \u2022 Each approved leave type that generates 0.0 points "
    "(FMLA, ADA, PTO/vacation, jury duty, military/USERRA, bereavement, workers comp, STD, "
    "and any other named in the policy)\n"
    "  \u2022 Each perfect attendance reduction milestone "
    "(threshold = consecutive clean days: 30, 60, 90, 365; points = negative value)\n"
    "  \u2022 Each corrective action level "
    "(threshold = lower bound of the point range; points = 0.0 \u2014 these are trigger rules, "
    "not point-generating occurrences)\n"
    "  Do NOT merge multiple violation types into a single rule row.\n\n"
    "Value derivation:\n"
    "  \u2022 id (organization, region, policy): generate a lower-kebab-case slug "
    "(e.g. 'acme-manufacturing', 'hr-att-2024-001')\n"
    "  \u2022 region.code: if not stated explicitly in the policy, derive a placeholder "
    "(e.g. 'AMC-PLANT-US') and note it must be resolved from facility master data\n"
    "  \u2022 region.timezone: infer from shift schedule context or mark as a placeholder "
    "(e.g. 'America/Detroit') to be confirmed from HRIS\n"
    "  \u2022 region.labor_laws: collect all referenced statutes as comma-separated TEXT "
    "(e.g. 'FMLA, ADA, USERRA, Workers Compensation')\n"
    "  \u2022 rule.condition: describe the HRIS/system condition that triggers this rule "
    "in plain English \u2014 do not write SQL\n"
    "  \u2022 rule.threshold: occurrence rules = 1; corrective action levels = lower bound "
    "of the point range; perfect attendance = consecutive clean days\n"
    "  \u2022 rule.points: positive for violations; 0.0 for approved-leave and CA-trigger rows; "
    "NEGATIVE for perfect-attendance reductions (e.g. -0.5, -1.0, -1.5, -2.0)\n\n"
    "Natural keys for existence checks (where_filters):\n"
    "  \u2022 organization: name OR code\n"
    "  \u2022 region: code\n"
    "  \u2022 organization_region: composite (organization_id, region_id)\n"
    "  \u2022 policy: name AND organization_id\n"
    "  \u2022 rule: policy_id AND name\n\n"
    "Planning rules:\n"
    "1. List tables_required in INSERT order (organization first, rule last).\n"
    "2. List joins as FK dependency chains: child table (left) \u2192 parent table (right); "
    "condition = FK column = PK column; join_type = 'INNER'.\n"
    "3. List where_filters: one existence check per table using the natural keys above. "
    "Set source_keyword to the extracted keyword that provides the natural key value.\n"
    "4. Put ALL column-to-keyword mappings and ALL rule row enumerations in cte_description. "
    "Do NOT put column mappings only in the purpose field of tables_required.\n"
    "5. Leave grouping_columns, aggregations, having_filters, and output_columns empty.\n\n"
    f"# FPI Schema\n\n```json\n{json.dumps(FPI_SCHEMA, indent=2)}\n```"
)

plan_model = model.with_structured_output(SQLQueryPlan)

PLAN_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content=PLAN_SYSTEM_MSG),
    HumanMessagePromptTemplate.from_template(
        "Devise an ingestion plan for the following keywords extracted from the "
        "attendance policy document. The plan must describe how to INSERT this "
        "policy's data into the FPI configuration tables. Do NOT write SQL.\n\n"
        "## Extracted Keywords\n\n{keywords}\n\n"
        "## Policy Document (for context)\n\n{pdf_text}"
    ),
])

print("PLAN_PROMPT ready")

In [ ]:
# ── _format_plan: plain-text serialiser for SQLQueryPlan ─────────────────────
def _format_plan(plan: SQLQueryPlan) -> str:
    lines = []
    lines.append(f"OBJECTIVE\n  {plan.objective}\n")

    if plan.tables_required:
        lines.append("TABLES")
        for t in plan.tables_required:
            lines.append(f"  [{t.alias}] {t.table}")
            lines.append(f"       purpose: {t.purpose}")
        lines.append("")

    if plan.joins:
        lines.append("JOINS")
        for j in plan.joins:
            lines.append(f"  {j.join_type} JOIN {j.right} ON {j.condition}")
        lines.append("")

    if plan.where_filters:
        lines.append("WHERE FILTERS")
        for f in plan.where_filters:
            lines.append(f"  \u2022 {f.description}")
            lines.append(f"    expr   : {f.expression}")
            lines.append(f"    keyword: {f.source_keyword}")
        lines.append("")

    if plan.uses_cte:
        lines.append("CTE")
        lines.append(f"  {plan.cte_description or '(no description)'}")

    return "\n".join(lines)


# ── Run: Step 3b — Query Plan Generation (RAG context) ────────────────────────
plan_data = None

if keyword_data:
    try:
        plan_data = (PLAN_PROMPT | plan_model).invoke(
            {
                "keywords": _format_keywords(keyword_data),
                "pdf_text": rag_context,
            },
            config={"callbacks": [metrics_cb]},
        )
        print(_format_plan(plan_data))
        print(f"Confidence: {plan_data.confidence_score}")
    except Exception as e:
        print(f"Plan generation error: {e}")
        plan_data = None
else:
    print("Run Step 3a first (keyword_data is None).")

## 7. LangGraph Agent Pipeline (RAG Edition)

Full pipeline with RAG retrieval as the first node.

**Graph:** `START → node_retrieve → node_extract → node_plan → END`

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# Structured output model variants for graph nodes
lg_kw_model   = model.with_structured_output(KeywordExtraction)
lg_plan_model = model.with_structured_output(SQLQueryPlan)

# ── Graph State ───────────────────────────────────────────────────────────────
class FPIState(TypedDict):
    pdf_text:      str
    rag_context:   str | None                # RAG-retrieved chunks from ChromaDB
    keywords:      KeywordExtraction | None
    sql_plan:      SQLQueryPlan | None
    error:         str | None

# ── RAG retrieval queries ─────────────────────────────────────────────────────
RAG_RETRIEVAL_QUERIES = [
    "attendance policy rules violations points thresholds escalation corrective action",
    "organization company employer region location jurisdiction labor laws",
    "policy name effective date scope accrual model rolling window",
    "excused unexcused absence tardy no-call no-show early departure",
    "perfect attendance reduction milestone FMLA ADA PTO leave approved",
]

# ── Node: Retrieve (RAG) ──────────────────────────────────────────────────────
def node_retrieve(state: FPIState) -> dict:
    """Chunk, embed, and retrieve relevant policy sections via ChromaDB."""
    text = state["pdf_text"]

    # Chunk
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_text(text)

    # Embed into ephemeral ChromaDB (in-memory)
    vs = Chroma.from_texts(texts=chunks, embedding=embeddings, collection_name="rag_pipeline")

    # Retrieve with multiple schema-aware queries (deduplicated)
    seen = set()
    relevant = []
    for q in RAG_RETRIEVAL_QUERIES:
        docs = vs.similarity_search(q, k=8)
        for doc in docs:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                relevant.append(doc.page_content)

    context = "\n\n---\n\n".join(relevant)
    print(f"  [node_retrieve] {len(chunks)} chunks embedded, {len(relevant)} unique retrieved, {len(context)} chars")
    return {"rag_context": context}

# ── Node: Extract Keywords ────────────────────────────────────────────────────
def node_extract(state: FPIState) -> dict:
    """Categorise policy keywords from RAG-retrieved context."""
    context = state.get("rag_context") or state["pdf_text"][:4000]
    result = (KEYWORD_PROMPT | lg_kw_model).invoke({"pdf_text": context})
    # Enforce realistic confidence
    if result.confidence_score >= 1.0:
        result.confidence_score = 0.85
    result.confidence_score = round(min(result.confidence_score, 0.95), 2)
    return {"keywords": result}

# ── Node: Generate Query Plan ─────────────────────────────────────────────────
def node_plan(state: FPIState) -> dict:
    """Produce a structured SQLQueryPlan from keywords + RAG context."""
    if not state.get("keywords"):
        return {"error": "node_plan: no keywords in state"}
    context = state.get("rag_context") or state["pdf_text"][:2000]
    result = (PLAN_PROMPT | lg_plan_model).invoke(
        {
            "keywords": _format_keywords(state["keywords"]),
            "pdf_text": context,
        },
    )
    # Enforce realistic confidence
    has_placeholders = bool(
        result.cte_description and "placeholder" in result.cte_description.lower()
    )
    if result.confidence_score >= 1.0:
        result.confidence_score = 0.82
    if has_placeholders and result.confidence_score > 0.85:
        result.confidence_score = 0.85
    result.confidence_score = round(min(max(result.confidence_score, 0.0), 0.95), 2)
    return {"sql_plan": result}

# ── Build Graph ───────────────────────────────────────────────────────────────
builder = StateGraph(FPIState)
builder.add_node("node_retrieve", node_retrieve)
builder.add_node("node_extract", node_extract)
builder.add_node("node_plan", node_plan)
builder.add_edge("__start__", "node_retrieve")
builder.add_edge("node_retrieve", "node_extract")
builder.add_edge("node_extract", "node_plan")
builder.add_edge("node_plan", "__end__")
fpi_rag_agent = builder.compile()

print("LangGraph RAG agent compiled: START \u2192 node_retrieve \u2192 node_extract \u2192 node_plan \u2192 END")

In [ ]:
# ── Run LangGraph RAG Agent ───────────────────────────────────────────────────
import time as _time

_t0 = _time.time()
lg_result = fpi_rag_agent.invoke(
    {
        "pdf_text": pdf_text,
        "rag_context": None,
        "keywords": None,
        "sql_plan": None,
        "error": None,
    },
    config={"callbacks": [metrics_cb]},
)
_elapsed = _time.time() - _t0

print(f"\n{'=' * 60}")
print(f"RAG PIPELINE COMPLETE \u2014 {_elapsed:.1f}s total")
print(f"{'=' * 60}")

if lg_result.get("error"):
    print(f"\nERROR: {lg_result['error']}")
else:
    kw = lg_result["keywords"]
    plan = lg_result["sql_plan"]
    print(f"\nRAG context: {len(lg_result.get('rag_context', '') or '')} chars")
    print(f"Keywords: {sum(len(getattr(kw, f, [])) for f in _KEYWORD_FIELDS)} terms (confidence {kw.confidence_score})")
    print(f"Plan: {plan.objective[:80]}... (confidence {plan.confidence_score})")
    print(f"\n--- Full Plan ---\n")
    print(_format_plan(plan))

## 8. Performance Comparison

In [ ]:
# ── Session Summary ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SESSION METRICS")
print("=" * 60)

total_tokens = 0
total_latency = 0
for i, rec in enumerate(metrics_cb.records):
    tokens = rec.get("total_tokens") or 0
    latency = rec.get("latency_ms", 0)
    total_tokens += tokens
    total_latency += latency
    print(f"  Call {i+1}: {latency:,.0f} ms | {tokens:,} tokens")

print(f"\n  TOTAL: {total_latency:,.0f} ms ({total_latency/1000:.1f}s) | {total_tokens:,} tokens")
print(f"\n  BASELINE (01_pdf_to_db_query): 316,700 ms (316.7s) | 36,000 tokens")
if total_latency > 0:
    speedup = 316700 / total_latency
    token_reduction = (36000 - total_tokens) / 36000 * 100
    print(f"  SPEEDUP: {speedup:.1f}x faster | {token_reduction:.0f}% fewer tokens")